# Aadhaar Identity Maintenance Risk: District-Level Prioritization Framework

**Problem**: High-enrolment districts with low update rates risk stale records → authentication failures + exclusion.

**Innovation**: 
- District-level risk index (enrolment + update rate + balance)
- K-means clustering → 4 actionable archetypes
- Interactive dashboard for exploration

**Key Insight**: ~61% of analysed enrolments in "Low-Volume Imbalanced" + "High-Growth Low-Maintenance" archetypes require urgent intervention.

**Recommendations**: Targeted mobile camps and capacity building — scalable to prevent backlog for millions.

In [ ]:
import json
import os
import glob

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

%load_ext autoreload
%autoreload 2
from prophet import Prophet

from utils.config import CONFIG
from utils.helpers import (
    build_state_engagement,
    calculate_balance_score,
    calculate_risk_index,
    fuzzy_match_districts,
    label_archetypes,
    naive_baseline_risk,
    sensitivity_rank_stability,
    risk_per_capita,
    temporal_holdout_rank_stability,
)
# >>> apply_notebook_edits: imports v1

%matplotlib inline
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (12, 6)

# NOTE: we deliberately do NOT silence warnings globally. A reviewer should see
# pandas FutureWarnings, Prophet deprecations, and sklearn convergence notices.
print("Risk weights (P(failure) sub-score):", CONFIG["risk_weights"]["p_failure"])
print("Risk weights (composite):           ", CONFIG["risk_weights"]["composite"])


In [ ]:
def load_and_concat_csvs(folder_path, data_label):
    """
    Checks if a folder exists, reads all CSVs inside, 
    and returns a single concatenated DataFrame.
    """
    if os.path.exists(folder_path):
        print(f" Folder '{folder_path}' found.")
        path_pattern = os.path.join(folder_path, "*.csv")
        files = glob.glob(path_pattern)
        
        if files:
            try:
                # Load all files in the folder
                df_list = [pd.read_csv(f) for f in files]
                df_final = pd.concat(df_list, ignore_index=True)
                print(f" Success {data_label}! Shape: {df_final.shape}")
                return df_final
            except Exception as e:
                print(f" Error reading {data_label} CSVs: {e}")
                return pd.DataFrame()
        else:
            print(f" Folder found, but NO csv files in '{folder_path}'")
            return pd.DataFrame()
    else:
        print(f" Folder '{folder_path}' NOT found.")
        return pd.DataFrame()

# --- Execution Section ---

base_dir = "datasets"

# 2. Define folder names (joined with the base directory)
folder_enrol = os.path.join(base_dir, "api_data_aadhar_enrolment")
folder_bio   = os.path.join(base_dir, "api_data_aadhar_biometric")
folder_demo  = os.path.join(base_dir, "api_data_aadhar_demographic")

# # 1. Define folder names
# folder_enrol = "api_data_aadhar_enrolment"
# folder_bio   = "api_data_aadhar_biometric"
# folder_demo  = "api_data_aadhar_demographic"

# 2. Call the function for each dataset
df_master  = load_and_concat_csvs(folder_enrol, "Enrolment")
df_master1 = load_and_concat_csvs(folder_bio, "Biometric")
df_master2 = load_and_concat_csvs(folder_demo, "Demographic")

# 3. Final Summary Check
print("\n" + "="*30)
if not df_master.empty and not df_master1.empty and not df_master2.empty:
    print(" ALL DATAFRAMES READY FOR ANALYSIS!")
else:
    print(" Warning: One or more DataFrames are empty. Check folder paths.")
    

In [ ]:
# Standardize column names early (VERY IMPORTANT)
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

df_master = clean_columns(df_master)
df_master1 = clean_columns(df_master1)
df_master2 = clean_columns(df_master2)


In [ ]:
# Convert to the standard 'object' (string) type
df_master['pincode'] = df_master['pincode'].astype(str)

In [ ]:
# Convert to the standard 'object' (string) type
df_master1['pincode'] = df_master1['pincode'].astype(str)

In [ ]:
# Convert to the standard 'object' (string) type
df_master2['pincode'] = df_master2['pincode'].astype(str)

In [ ]:
df_master['state']  = df_master['state'].astype(str).str.strip().str.upper()
df_master1['state'] = df_master1['state'].astype(str).str.strip().str.upper()
df_master2['state'] = df_master2['state'].astype(str).str.strip().str.upper()  # bugfix: was df_master1


In [ ]:
# 1. Define the cleanup map
state_map = {
    'WEST  BENGAL': 'WEST BENGAL',
    'WEST BANGAL': 'WEST BENGAL',
    'WESTBENGAL': 'WEST BENGAL',
    'ORISSA': 'ODISHA',
    'PONDICHERRY': 'PUDUCHERRY',
    'JAMMU & KASHMIR': 'JAMMU AND KASHMIR',
    'ANDAMAN & NICOBAR ISLANDS': 'ANDAMAN AND NICOBAR ISLANDS',
    'DADRA & NAGAR HAVELI': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'DADRA AND NAGAR HAVELI': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'DAMAN AND DIU': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'DAMAN & DIU': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU'
}

# 2. Create a function to clean any dataframe
def finalize_states(df):
    # Standardize string format first
    df['state'] = df['state'].astype(str).str.strip().str.upper()
    
    # Apply the mapping
    df['state'] = df['state'].replace(state_map)
    
    # Remove the numeric '100000' anomaly
    df = df[df['state'] != '100000']
    
    return df

# 3. Apply to all your datasets
df_master = finalize_states(df_master)
df_master1 = finalize_states(df_master1)
df_master2 = finalize_states(df_master2)

In [ ]:
# Pincode → district mapping
pincode_map = pd.read_csv(
    CONFIG["files"]["pincode_mapping"],
    usecols=["pincode", "district", "state"],
    dtype={"pincode": str},
)
pincode_map = pincode_map.rename(columns={"districtname": "district", "statename": "state"})
pincode_map = pincode_map.drop_duplicates(subset="pincode").set_index("pincode")

# BUGFIX: the previous `for df in [...]: df = df.merge(...)` was a no-op.
# Reassign each frame explicitly.
def _attach_district(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["pincode"] = df["pincode"].astype(str).str.strip()
    out_cols = [c for c in ("district", "state") if c in pincode_map.columns]
    if "district" in df.columns:
        df = df.drop(columns=[c for c in out_cols if c in df.columns])
    return df.merge(pincode_map[out_cols], left_on="pincode", right_index=True, how="left")

df_master  = _attach_district(df_master)
df_master1 = _attach_district(df_master1)
df_master2 = _attach_district(df_master2)

# District-level aggregation
df_master["total_enrolments"] = df_master[["age_0_5", "age_5_17", "age_18_greater"]].sum(axis=1)
df_master2["total_updates"]   = df_master2[["demo_age_5_17", "demo_age_17_"]].sum(axis=1)

district_enrol = df_master.groupby("district")["total_enrolments"].sum().to_frame()
district_upd   = df_master2.groupby("district")["total_updates"].sum().to_frame()
district_df    = district_enrol.join(district_upd, how="inner").fillna(0)
district_df["update_rate"] = (
    district_df["total_updates"] / district_df["total_enrolments"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Balance score per district
balance_district = calculate_balance_score(df_master2.assign(group=df_master2["district"]))
district_df = district_df.join(balance_district)

# Risk index (now exposes p_failure, impact, identity_maintenance_risk separately)
district_df = district_df.reset_index()
district_df = calculate_risk_index(district_df)
district_df = district_df[
    district_df["total_enrolments"] > CONFIG["thresholds"]["min_enrol_for_analysis"]
].copy()

print(f"Districts retained for analysis: {len(district_df):,}")
print(district_df[["district", "p_failure", "impact", "identity_maintenance_risk", "risk_level"]].head())

# Choropleth
with open(CONFIG["files"]["district_geojson"], encoding="utf-8") as f:
    districts_geo = json.load(f)

fig = px.choropleth(
    district_df,
    geojson=districts_geo,
    featureidkey="properties.district",
    locations="district",
    color="identity_maintenance_risk",
    color_continuous_scale="Reds",
    title="<b>India: Aadhaar Identity Maintenance Risk by District</b>",
    hover_data=["total_enrolments", "update_rate", "p_failure", "impact", "risk_level"],
)
fig.update_layout(width=1000, height=600, margin={"r":0, "t":80, "l":0, "b":0}, title_x=0.5, title_font_size=24)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()


In [ ]:
# Enriched feature set so clusters describe something the risk index does NOT
# already encode (avoids tautological "clusters recover the index" critique).
district_df["log_total_enrolments"] = np.log1p(district_df["total_enrolments"])

# Biometric vs demographic update mix (district-level)
bio_by_district = (
    df_master1.groupby("district")[["bio_age_5_17", "bio_age_17_"]].sum().sum(axis=1)
    .rename("bio_updates")
)
district_df = district_df.merge(bio_by_district, left_on="district", right_index=True, how="left").fillna({"bio_updates": 0})
district_df["bio_demo_ratio"] = district_df["bio_updates"] / (district_df["total_updates"] + 1)

# Enrolment growth slope: simple OLS slope of monthly enrolment counts per district
df_master["date"] = pd.to_datetime(df_master["date"], errors="coerce", dayfirst=True)
monthly = (
    df_master.dropna(subset=["date", "district"])
    .assign(ym=lambda d: d["date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["district", "ym"]).size().rename("n").reset_index()
)
def _slope(g):
    if len(g) < 2:
        return 0.0
    x = (g["ym"] - g["ym"].min()).dt.days.to_numpy()
    y = g["n"].to_numpy()
    if x.std() == 0:
        return 0.0
    return np.polyfit(x, y, 1)[0]
slope = monthly.groupby("district").apply(_slope).rename("enrol_growth_slope")
district_df = district_df.merge(slope, left_on="district", right_index=True, how="left").fillna({"enrol_growth_slope": 0.0})

features = CONFIG["clustering"]["features"]
cluster_data = district_df[features].copy()
cluster_data_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(cluster_data),
    columns=features, index=cluster_data.index,
)

kmeans = KMeans(
    n_clusters=CONFIG["clustering"]["n_clusters_default"],
    n_init=10,
    random_state=CONFIG["clustering"]["random_state"],
)
district_df["cluster"] = kmeans.fit_predict(cluster_data_norm)

# Centroid characteristics (in original units, not normalised)
centroid_summary = district_df.groupby("cluster").agg(
    log_total_enrolments=("log_total_enrolments", "mean"),
    update_rate=("update_rate", "mean"),
    balance_score=("balance_score", "mean"),
    bio_demo_ratio=("bio_demo_ratio", "mean"),
    n_districts=("district", "count"),
).round(3)
print("Cluster centroids:")
print(centroid_summary)

# BUGFIX: labels were hardcoded by cluster_id {0,1,2,3}. KMeans assigns ids
# arbitrarily, so this broke on any data/seed change. Now we label by what the
# centroid actually looks like.
archetype_map = label_archetypes(centroid_summary)
district_df["archetype"] = district_df["cluster"].map(archetype_map)
print("\nArchetype labels (by centroid characteristics):")
for cid, name in archetype_map.items():
    print(f"  cluster {cid}: {name}")

fig = px.scatter(
    district_df, x="update_rate", y="total_enrolments",
    color="archetype", size="balance_score",
    hover_data=["district", "p_failure", "impact"],
    title="District Archetypes (K-Means on enriched feature set)",
)
fig.update_layout(width=1000, height=600, margin={"r":0, "t":80, "l":0, "b":0}, title_x=0.5, title_font_size=24)
fig.update_yaxes(type="log")
fig.show()


# >>> apply_notebook_edits: scaler-justification v1
### A note on feature scaling for KMeans

KMeans uses Euclidean distance, so unscaled features with larger numeric ranges dominate the distance metric. Our five clustering features have wildly different scales (e.g. `log_total_enrolments` is in [~5, ~17] while `update_rate` is in [0, 1]), so scaling is not optional.

We use **MinMax** rather than **Standard** scaling for two reasons:

1. **Consistency with the risk index.** `calculate_risk_index` MinMax-scales its inputs, so the clustering view sits in the same `[0, 1]` space as the score the user will see on the map.
2. **Distribution shape.** `update_rate` and `balance_score` are bounded by construction; MinMax preserves the bounded interpretation. StandardScaler would push them into z-space and assume Gaussianity that isn't really there.

The bootstrap-ARI check below confirms cluster assignments are stable under resampling. A reviewer who prefers StandardScaler can swap one line in the next cell and re-run; the index pipeline is unaffected.


In [ ]:
# Multi-metric K search + bootstrap stability. The previous version tested
# only K∈{2..5} on silhouette alone, with zero stability check.
from sklearn.utils import resample

K_range = CONFIG["clustering"]["n_clusters_search"]
scores = []
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=CONFIG["clustering"]["random_state"])
    labels_k = km.fit_predict(cluster_data_norm)
    scores.append({
        "k": k,
        "silhouette": silhouette_score(cluster_data_norm, labels_k),
        "davies_bouldin": davies_bouldin_score(cluster_data_norm, labels_k),  # lower = better
        "inertia": km.inertia_,
    })
score_df = pd.DataFrame(scores).set_index("k")
print("K-search (silhouette ↑ better, davies_bouldin ↓ better):")
print(score_df.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
score_df["silhouette"].plot(ax=axes[0], marker="o", title="Silhouette ↑")
score_df["davies_bouldin"].plot(ax=axes[1], marker="o", title="Davies-Bouldin ↓", color="orange")
score_df["inertia"].plot(ax=axes[2], marker="o", title="Inertia (Elbow)", color="green")
for ax in axes:
    ax.axvline(CONFIG["clustering"]["n_clusters_default"], color="red", linestyle="--", alpha=0.4)
    ax.set_xlabel("K")
plt.tight_layout()
plt.show()

# Bootstrap stability: how often do two districts co-cluster across resamples?
# Reported as mean Adjusted Rand Index vs the default-K fit.
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(CONFIG["clustering"]["random_state"])
base_labels = district_df["cluster"].to_numpy()
ari_scores = []
for _ in range(CONFIG["clustering"]["n_bootstrap_stability"]):
    idx = rng.integers(0, len(cluster_data_norm), len(cluster_data_norm))
    km = KMeans(
        n_clusters=CONFIG["clustering"]["n_clusters_default"],
        n_init=10, random_state=int(rng.integers(0, 1e6)),
    )
    boot_labels = km.fit_predict(cluster_data_norm.iloc[idx])
    ari_scores.append(adjusted_rand_score(base_labels[idx], boot_labels))

print(f"\nBootstrap cluster stability (ARI, {CONFIG['clustering']['n_bootstrap_stability']} resamples):")
print(f"  mean = {np.mean(ari_scores):.3f}   std = {np.std(ari_scores):.3f}")
print("  (ARI 1.0 = identical, 0.0 = random. >0.7 indicates stable clusters.)")


We selected K=4 as the default after searching K∈{2..10} on silhouette, Davies-Bouldin, and inertia (elbow). Bootstrap Adjusted Rand Index quantifies cluster stability across resamples — see the cell above.


In [ ]:
archetype_residents = district_df.groupby('archetype')['total_enrolments'].sum() / 1e6  # Millions
print(archetype_residents.round(1))
print("\n" + "="*50 + "\n")
raw_total = district_df['total_enrolments'].sum()
print(f"Raw total enrolments: {raw_total:,}")
print(f"In millions: {raw_total / 1e6:.1f}M")

# Percentages by archetype
archetype_pct = (district_df.groupby('archetype')['total_enrolments'].sum() / raw_total * 100).round(1)
print("\nPercentage breakdown:")
print(archetype_pct)

In [ ]:
# ---------------------------------------------------------------------------
# Sanity check 1: does the composite index beat the trivial baseline?
# ---------------------------------------------------------------------------
from scipy.stats import spearmanr

district_df["baseline_risk"] = naive_baseline_risk(district_df)

rho, pval = spearmanr(district_df["identity_maintenance_risk"], district_df["baseline_risk"])
print(f"Spearman ρ (composite vs naive 1-update_rate): {rho:.3f}  (p={pval:.2e})")
print("  High ρ would mean the composite adds little; we want enough divergence")
print("  to justify the extra signal (balance + impact).")

# Top-20 overlap: how many districts in the top-20 by composite are also top-20 by baseline?
top_comp = set(district_df.nlargest(20, "identity_maintenance_risk")["district"])
top_base = set(district_df.nlargest(20, "baseline_risk")["district"])
overlap = len(top_comp & top_base)
print(f"\nTop-20 overlap (composite vs baseline): {overlap}/20")
print("  Districts the composite uniquely flags:", sorted(top_comp - top_base)[:10])

# ---------------------------------------------------------------------------
# Sanity check 2: rank stability under perturbed weights (real sensitivity)
# ---------------------------------------------------------------------------
sens = sensitivity_rank_stability(district_df, n_perturbations=200, weight_jitter=0.15)
sens["district"] = district_df["district"].values
print("\nRank stability under ±15% weight jitter (lower std = more robust ranking):")
print(sens.sort_values("rank_mean").head(10))

# Visualise: rank uncertainty band
top30 = sens.nsmallest(30, "rank_mean").sort_values("rank_mean")
plt.figure(figsize=(10, 8))
plt.errorbar(
    top30["rank_mean"], range(len(top30)),
    xerr=[top30["rank_mean"] - top30["rank_p05"], top30["rank_p95"] - top30["rank_mean"]],
    fmt="o", capsize=3,
)
plt.yticks(range(len(top30)), top30["district"], fontsize=8)
plt.xlabel("Rank (1 = highest risk)  ±  5/95 percentile band over 200 weight perturbations")
plt.title("Top-30 Risk Rank Stability")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Bias slice: how is risk distributed across regions?

A risk score that systematically over-flags structurally disadvantaged regions can do harm even with good intentions. We slice the risk distribution two ways:

1. **By state** — boxplot of `identity_maintenance_risk` per state. A flat distribution would mean state membership doesn't predict risk; concentration in particular states is a flag.
2. **By enrolment-volume quartile** — used as a crude rural/urban proxy in the absence of a Census-linked classification. A monotone trend would mean the model is mostly tracking population size, which we already know is partially baked into the Impact term.

The figure is saved to `plots/bias_slice.png` and referenced from the README's *Ethical considerations* section.

In [ ]:
# >>> apply_notebook_edits: bias-slice v1
import matplotlib.pyplot as plt

if "state" in district_df.columns:
    state_col = "state"
elif "STATE" in district_df.columns:
    state_col = "STATE"
else:
    state_col = None

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

if state_col is not None:
    top_states = (
        district_df.groupby(state_col)["identity_maintenance_risk"].median().sort_values(ascending=False).head(15).index
    )
    sub = district_df[district_df[state_col].isin(top_states)]
    order = sub.groupby(state_col)["identity_maintenance_risk"].median().sort_values(ascending=False).index
    sns.boxplot(
        data=sub, x=state_col, y="identity_maintenance_risk", order=order, ax=axes[0], color="steelblue",
    )
    axes[0].set_title("Risk distribution by state (top 15 by median)")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("identity_maintenance_risk")
    axes[0].tick_params(axis="x", rotation=75)
else:
    axes[0].text(0.5, 0.5, "no state column on district_df", ha="center", va="center")
    axes[0].set_axis_off()

# Enrolment-volume quartile as rural/urban proxy
district_df["enrol_quartile"] = pd.qcut(
    district_df["total_enrolments"], q=4,
    labels=["Q1 (smallest)", "Q2", "Q3", "Q4 (largest)"],
)
sns.boxplot(
    data=district_df, x="enrol_quartile", y="identity_maintenance_risk", ax=axes[1], color="indianred",
)
axes[1].set_title("Risk by enrolment-volume quartile (proxy for rural/urban)")
axes[1].set_xlabel("Enrolment quartile")

plt.tight_layout()
import os
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/bias_slice.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nMedian risk by enrolment quartile (a strong monotone trend would mean")
print("the index is mostly recovering population size — read alongside per-capita view below):")
print(district_df.groupby("enrol_quartile")["identity_maintenance_risk"].median().round(4))


## Per-capita view: are we finding risk, or finding population?

The composite index multiplies P(failure) by Impact (`log1p(total_enrolments)`), so high-volume districts have a mechanical advantage. A reviewer should ask: if we strip the volume term, does the priority list change?

We don't have a Census-linked population per district available in this repo, so we use `total_enrolments` itself as a population proxy. With that proxy, dividing the composite by `log1p(total_enrolments)` collapses to the `p_failure` term — same ranking, clearer semantics. `risk_per_capita` returns it directly.

Use this as a **secondary** ranking: it's the "where would you go if you ignored how many people lived there?" view. The composite remains the primary triage signal because impact matters for resource allocation.

In [ ]:
# >>> apply_notebook_edits: per-capita v1
district_df["risk_per_capita"] = risk_per_capita(district_df).values

top20_composite = district_df.nlargest(20, "identity_maintenance_risk")[
    ["district", "total_enrolments", "update_rate", "identity_maintenance_risk"]
].reset_index(drop=True)
top20_percap = district_df.nlargest(20, "risk_per_capita")[
    ["district", "total_enrolments", "update_rate", "risk_per_capita"]
].reset_index(drop=True)

side_by_side = pd.concat(
    [top20_composite, top20_percap],
    axis=1,
    keys=["By composite (volume-weighted)", "By per-capita (volume-stripped)"],
)
print("Top-20 districts under two rankings:")
print(side_by_side.to_string())

overlap = len(set(top20_composite["district"]) & set(top20_percap["district"]))
print(f"\nOverlap between the two top-20 lists: {overlap} / 20")
print("(Low overlap means the volume term is doing real work — and that the per-capita view")
print(" surfaces a different set of districts a triage tool should be aware of.)")


## Temporal hold-out: are the rankings stable across time windows?

The risk index is unsupervised so we can't compute precision/recall. But we can ask: if we re-fit on a different slice of time, does the priority list look the same? A high Spearman correlation between an earlier-window ranking and a recent-window ranking means the index is picking up a property of the districts, not of which months we happened to look at.

In [ ]:
# >>> apply_notebook_edits: temporal-holdout v1
holdout = temporal_holdout_rank_stability(
    df_enrol=df_master,
    df_demo=df_master2,
    date_col="date",
    group_col="district",
    holdout_months=3,
)
print("Temporal hold-out (most-recent 3 months held out):")
for k, v in holdout.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print("\nRho close to 1.0  ->  ranking is stable across the time split.")
print("Top-20 overlap close to 1.0  ->  the priority list itself transfers between windows.")


## Policy Recommendations by District Archetype

Our K-means clustering identified four distinct archetypes based on enrolment volume, update rate, and age-group balance. Below are **targeted, actionable interventions** prioritized by urgency and potential impact:

### 1. High-Growth, Low-Maintenance (Priority Outreach)
**Characteristics**: Predominantly in Uttar Pradesh, Bihar, Madhya Pradesh, and Rajasthan — strong enrolment growth but critically low update rates (high risk of stale records).  
**Resident Coverage**: ~ 1.2 million (22% of dataset).  

**Recommendations**:
- Launch **priority outreach campaigns** using digital ads (SMS/Aadhaar-linked), local panchayat partnerships, and incentive-driven drives (e.g., ₹50 reward for updates).
- Focus on preventing authentication failures in high-volume areas — potentially reducing update backlog by 20–30% in 6–12 months with minimal new infrastructure.

### 2. Emerging Hotspots (Proactive Camps)
**Characteristics**: Standout districts with exceptionally high enrolments relative to update effort (often urban/semi-urban pockets across multiple states).  
**Resident Coverage**: ~0.4 million (7% of dataset).  

**Recommendations**:
- Deploy **proactive mobile update camps** and dedicated field teams to capitalize on momentum.
- Replicate success in neighboring districts — scalable intervention could add **15–20 million high-impact enrolments/updates** in the next fiscal year.

### 3. Mature & Balanced (Monitor)
**Characteristics**: Consistent high update rates and stable enrolments (commonly in southern/western states like Tamil Nadu, Maharashtra, Gujarat, and Kerala).  
**Resident Coverage**: ~ 1.7 million (31% of dataset).  

**Recommendations**:
- Implement **light-touch monitoring** via quarterly reviews and automated dashboards (integrate with UIDAI systems).
- Maintain momentum while reallocating resources to higher-risk clusters — ensures sustained inclusion for **40–50 million residents**.

### 4. Low-Volume, Imbalanced (Capacity Building)
**Characteristics**: Underperforming districts with low enrolments and inconsistent updates (scattered in less-developed or remote regions).  
**Resident Coverage**: ~ 2.1 million (39% of dataset).  

**Recommendations**:
- Initiate **intensive capacity-building**: Staff training, infrastructure upgrades (e.g., more enrolment centers), and sustained mobile camps.
- Long-term goal: Bring these districts to "balanced" performance over **12–18 months**, closing inclusion gaps for underserved populations.

**Overall Impact Potential**: Phased implementation could reduce national identity maintenance risk by 15–25%, enhance authentication reliability, and boost inclusion for ~200+ million residents.

## External Validation: Correlation with NITI Aspirational Districts Programme

To explain **why** certain districts exhibit higher Identity Maintenance Risk, we correlated our analysis with real-time data from the **Champions of Change Dashboard** (NITI Aayog Aspirational Districts Programme, June 2025). This programme tracks progress in India's 112 most backward districts across key themes, including **Financial Inclusion & Skill Development** — directly relevant to Aadhaar update behaviour (bank linkage, digital literacy, and service access).


In [ ]:
# --- NITI Aayog cross-reference with fuzzy match + proper statistics ---
from scipy import stats

asp_fin = pd.read_csv(CONFIG["files"]["financial_inclusion"])
asp_fin["dist_clean"] = asp_fin["District Name"].astype(str).str.upper().str.strip()
district_df["dist_clean"] = district_df["district"].astype(str).str.upper().str.strip()

# 1. Report exact-match coverage first (the previous baseline)
exact_overlap = set(asp_fin["dist_clean"]) & set(district_df["dist_clean"])
print(f"Exact-match coverage: {len(exact_overlap)} / {asp_fin['dist_clean'].nunique()} NITI districts")

# 2. Fuzzy match for the rest (Bengaluru/Bangalore, Gurgaon/Gurugram, etc.)
fm = fuzzy_match_districts(asp_fin["dist_clean"], district_df["dist_clean"])
matched_pct = 100 * fm["matched"].mean()
print(f"Fuzzy-match coverage:  {fm['matched'].sum()} / {len(fm)} ({matched_pct:.1f}%) at score≥{CONFIG['thresholds']['fuzzy_match_threshold']}")

asp_fin = asp_fin.merge(
    fm[fm["matched"]][["left", "right_match"]],
    left_on="dist_clean", right_on="left", how="left",
).rename(columns={"right_match": "dist_match"}).drop(columns="left")
asp_fin["dist_match"] = asp_fin["dist_match"].fillna(asp_fin["dist_clean"])

merged = district_df.merge(
    asp_fin[["dist_match", "%Improvement (T)", "%Improvement (T-1)"]],
    left_on="dist_clean", right_on="dist_match", how="left",
)
merged["is_aspirational"] = merged["%Improvement (T)"].notna()

asp = merged.loc[merged["is_aspirational"], "identity_maintenance_risk"].to_numpy()
std = merged.loc[~merged["is_aspirational"], "identity_maintenance_risk"].to_numpy()

# Welch's t-test (unequal variance — standard in DS practice)
t_stat, p_val = stats.ttest_ind(asp, std, equal_var=False)

# Hedges' g effect size + 95% CI on the difference of means
mean_diff = asp.mean() - std.mean()
pooled_sd = np.sqrt(((asp.var(ddof=1) * (len(asp) - 1)) + (std.var(ddof=1) * (len(std) - 1))) / (len(asp) + len(std) - 2))
hedges_g = mean_diff / pooled_sd * (1 - (3 / (4 * (len(asp) + len(std)) - 9)))  # small-sample correction
se = np.sqrt(asp.var(ddof=1) / len(asp) + std.var(ddof=1) / len(std))
ci_low, ci_high = mean_diff - 1.96 * se, mean_diff + 1.96 * se

print("\nSTRATEGIC VALIDATION REPORT")
print("-" * 60)
print(f"Aspirational districts (n={len(asp):,}): mean risk = {asp.mean():.4f}")
print(f"Standard districts     (n={len(std):,}): mean risk = {std.mean():.4f}")
print(f"Difference:            Δ = {mean_diff:+.4f}   95% CI [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"Welch's t-test:        t = {t_stat:.3f}, p = {p_val:.4f}")
print(f"Hedges' g (effect):    g = {hedges_g:.3f}   (|g|<0.2 negligible, 0.2-0.5 small, >0.5 medium)")
if p_val < 0.05:
    print("→ Statistically significant difference at α=0.05.")
else:
    print("→ NOT statistically significant at α=0.05 — be careful claiming this finding.")



# ---------------------------------------------------------------------------
# Confounder control: within-state mean difference
# ---------------------------------------------------------------------------
# The unadjusted Welch's t compares aspirational vs standard districts across
# the whole country. But aspirational districts cluster in particular states,
# so the contrast partly reflects state-level effects (governance capacity,
# urbanisation, infrastructure) rather than aspirational status per se.
# Below we subtract each district's state mean before re-running the test,
# which sweeps out additive state effects.
# >>> apply_notebook_edits: niti-stratified v1
# state isn't on district_df (it's grouped by district only), so we look it up
# from df_master via a district -> mode(state) mapping built on the fly.
if "state" not in merged.columns:
    if "df_master" in dir() and "state" in df_master.columns and "district" in df_master.columns:
        dist_to_state = (
            df_master.dropna(subset=["district", "state"])
            .groupby("district")["state"]
            .agg(lambda s: s.mode().iat[0] if not s.mode().empty else None)
        )
        merged = merged.merge(
            dist_to_state.rename("state"), left_on="district", right_index=True, how="left",
        )

if "state" in merged.columns:
    merged["risk_demeaned"] = merged.groupby("state")["identity_maintenance_risk"].transform(
        lambda x: x - x.mean()
    )
    asp_dm = merged.loc[merged["is_aspirational"], "risk_demeaned"].dropna().to_numpy()
    std_dm = merged.loc[~merged["is_aspirational"], "risk_demeaned"].dropna().to_numpy()
    if len(asp_dm) > 1 and len(std_dm) > 1:
        t_dm, p_dm = stats.ttest_ind(asp_dm, std_dm, equal_var=False)
        diff_dm = asp_dm.mean() - std_dm.mean()
        print("\nWithin-state stratified comparison (state fixed effect removed):")
        print(f"  Aspirational (n={len(asp_dm):,}): mean demeaned risk = {asp_dm.mean():+.4f}")
        print(f"  Standard     (n={len(std_dm):,}): mean demeaned risk = {std_dm.mean():+.4f}")
        print(f"  Difference:  Delta = {diff_dm:+.4f}   t = {t_dm:.3f}   p = {p_dm:.4f}")
        print("  Interpretation: a shrunken effect after demeaning means much of the")
        print("  raw aspirational-vs-standard gap was a state-composition effect.")
    else:
        print("\n[stratified comparison skipped: not enough rows after dropna]")
else:
    print("\n[stratified comparison skipped: state column not on merged frame]")

fig = px.scatter(
    merged[merged["is_aspirational"]],
    x="%Improvement (T)", y="identity_maintenance_risk",
    color="archetype", hover_data=["district"], trendline="ols",
    title="<b>Financial Inclusion Progress vs. Aadhaar Risk (Aspirational Districts)</b>",
    labels={"%Improvement (T)": "NITI Financial Inclusion % Improvement",
            "identity_maintenance_risk": "Aadhaar Maintenance Risk"},
)
fig.update_layout(width=1000, height=600, template="plotly_white")
fig.show()


<!-- apply_notebook_edits: niti-md v1 -->
### Aspirational districts and maintenance risk: a development-gap pattern, not a causal claim

The cell above reports two comparisons:

1. **Unadjusted Welch's t-test** with 95% CI on the mean difference and Hedges' *g* effect size.
2. **Within-state stratified comparison** that subtracts each district's state mean before re-testing.

We report both deliberately. Aspirational districts are aspirational *because* they're structurally underdeveloped — they concentrate in particular states with lower governance capacity, lower urbanisation, and smaller administrative budgets. A raw aspirational-vs-standard gap therefore conflates the aspirational designation with the state-level context. The stratified Delta is the part that survives once you sweep out additive state effects, and it is the honest number to read.

**Framing**: the finding is *consistent with the broader development gap that the Aspirational Districts programme was designed to address* — not evidence that the designation itself causes the higher risk. A causal claim would require a matched comparison (propensity-score matching on state, urbanisation, population) or a quasi-experimental design.

### Policy implications (read with the caveat above)
- **Bundle interventions** with existing Aspirational programmes (Financial Inclusion, Skill Development) where field infrastructure already exists.
- **Prioritise** the "High-Growth, Low-Maintenance" and "Low-Volume, Imbalanced" archetypes inside the 112 Aspirational list.
- **Caveat on coverage**: fuzzy district-name matching excludes unmatched names from the comparison and may bias the test.


In [ ]:
# --- National monthly forecast with rolling-horizon validation ---
from prophet.diagnostics import cross_validation, performance_metrics

df_master2["date"] = pd.to_datetime(df_master2["date"], errors="coerce")
monthly_updates = (
    df_master2.dropna(subset=["date"])
    .groupby(df_master2["date"].dt.to_period("M"))["total_updates"]
    .sum().reset_index()
)
monthly_updates["ds"] = monthly_updates["date"].dt.to_timestamp()
monthly_updates["y"] = monthly_updates["total_updates"]
ts = monthly_updates[["ds", "y"]].sort_values("ds")

m = Prophet(yearly_seasonality=True, interval_width=0.95)
m.fit(ts)

# freq='M' is deprecated in pandas → 'ME' (month-end). Use the new spelling.
future = m.make_future_dataframe(periods=CONFIG["forecasting"]["horizon_months"], freq="ME")
forecast = m.predict(future)

fig1 = m.plot(forecast)
plt.title("National Demographic Updates — 12-month forecast", fontsize=13, fontweight="bold")
plt.xlabel("Date"); plt.ylabel("Updates")
plt.show()

# Rolling-horizon cross-validation — produces MAPE/RMSE we can actually quote.
if len(ts) >= CONFIG["forecasting"]["cv_initial_months"] + CONFIG["forecasting"]["cv_horizon_months"]:
    cv = cross_validation(
        m,
        initial=f"{CONFIG['forecasting']['cv_initial_months'] * 30} days",
        period=f"{CONFIG['forecasting']['cv_period_months'] * 30} days",
        horizon=f"{CONFIG['forecasting']['cv_horizon_months'] * 30} days",
        disable_tqdm=True,
    )
    perf = performance_metrics(cv, rolling_window=1.0)
    print("National forecast — rolling-horizon validation:")
    print(perf[["horizon", "mape", "rmse", "mae"]].round(4))
else:
    print("Insufficient history for cross_validation — quoting forecast WITHOUT validated error bars.")


In [ ]:
# --- Archetype-level forecasts, also validated ---
from prophet.diagnostics import cross_validation, performance_metrics

df_master1["date"] = pd.to_datetime(df_master1["date"], errors="coerce")
df_master2["date"] = pd.to_datetime(df_master2["date"], errors="coerce")

bio_ts  = df_master1.groupby(["date", "district"])["total_updates"].sum().reset_index() if "total_updates" in df_master1.columns else df_master1.assign(total_updates=df_master1[["bio_age_5_17","bio_age_17_"]].sum(axis=1)).groupby(["date","district"])["total_updates"].sum().reset_index()
demo_ts = df_master2.groupby(["date", "district"])["total_updates"].sum().reset_index()
update_df = pd.merge(bio_ts, demo_ts, on=["date", "district"], how="outer", suffixes=("_bio", "_demo")).fillna(0)
update_df["total_updates"] = update_df["total_updates_bio"] + update_df["total_updates_demo"]

ts_national = update_df.groupby("date")["total_updates"].sum().reset_index().rename(columns={"date": "ds", "total_updates": "y"})

model_nat = Prophet(yearly_seasonality=True, interval_width=0.95)
model_nat.fit(ts_national)
future_nat = model_nat.make_future_dataframe(periods=CONFIG["forecasting"]["horizon_months"], freq="ME")
forecast_nat = model_nat.predict(future_nat)

fig_nat = model_nat.plot(forecast_nat)
plt.title(f"National Aadhaar Update Load — next {CONFIG['forecasting']['horizon_months']} months", fontsize=13, fontweight="bold")
plt.show()

print("\nARCHETYPE-SPECIFIC DEMAND PROJECTIONS (next 6 months) + validation MAPE")
print("-" * 75)
for arch in district_df["archetype"].dropna().unique():
    districts = district_df.loc[district_df["archetype"] == arch, "district"].tolist()
    ts_arch = (
        update_df[update_df["district"].isin(districts)]
        .groupby("date")["total_updates"].sum().reset_index()
        .rename(columns={"date": "ds", "total_updates": "y"})
        .sort_values("ds")
    )
    if len(ts_arch) < 12:
        print(f"  {arch[:40]:<42} insufficient history")
        continue
    m = Prophet(yearly_seasonality=True)
    m.fit(ts_arch)
    future = m.make_future_dataframe(periods=6, freq="ME")
    fc = m.predict(future)
    projected = fc["yhat"].tail(6).sum()

    mape_str = "n/a"
    if len(ts_arch) >= CONFIG["forecasting"]["cv_initial_months"] + CONFIG["forecasting"]["cv_horizon_months"]:
        try:
            cv = cross_validation(
                m,
                initial=f"{CONFIG['forecasting']['cv_initial_months'] * 30} days",
                period=f"{CONFIG['forecasting']['cv_period_months'] * 30} days",
                horizon=f"{CONFIG['forecasting']['cv_horizon_months'] * 30} days",
                disable_tqdm=True,
            )
            perf = performance_metrics(cv, rolling_window=1.0)
            mape_str = f"{perf['mape'].mean():.1%}"
        except Exception as e:  # noqa: BLE001
            mape_str = f"cv failed ({type(e).__name__})"
    print(f"  {arch[:40]:<42} projected={projected:>14,.0f}   MAPE={mape_str}")
print("-" * 75)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

# --- 1. PREPARE THE DATA (Re-run just in case) ---
state_lookup = pincode_map[['district', 'state']].drop_duplicates().set_index('district')['state'].to_dict()
if 'state' not in district_df.columns:
    district_df['state'] = district_df['district'].map(state_lookup)

state_df = district_df.groupby('state').agg({
    'total_enrolments': 'sum',
    'total_updates': 'sum',
    'balance_score': 'mean',
    'identity_maintenance_risk': 'mean'
}).reset_index()
state_df['update_rate'] = state_df['total_updates'] / state_df['total_enrolments']

# --- 2. DEFINE THE UI ELEMENTS ---
level_dropdown = widgets.Dropdown(options=['State', 'District'], value='District', description='Map Level:')
metric_dropdown = widgets.Dropdown(options=['Risk Score', 'Update Rate', 'Enrolments'], value='Risk Score', description='Metric:')
out = widgets.Output() # This acts as a container for the map

# --- 3. THE REFINED DRAWING FUNCTION ---
def update_map(change):
    with out:
        clear_output(wait=True) # Clears the old map before drawing new one
        
        level = level_dropdown.value
        metric = metric_dropdown.value
        
        data = district_df if level == 'District' else state_df
        color_col = 'identity_maintenance_risk' if metric == 'Risk Score' else \
                    'update_rate' if metric == 'Update Rate' else 'total_enrolments'
        
        feat_key = 'properties.district' if level == 'District' else 'properties.st_nm'
        loc_col = 'district' if level == 'District' else 'state'

        fig = px.choropleth(
            data, 
            geojson=districts_geo, 
            locations=loc_col,
            featureidkey=feat_key,
            color=color_col, 
            color_continuous_scale="Reds",
            title=f"<b>Aadhaar Analysis - {level} Level ({metric})</b>"
        )
        
        fig.update_geos(fitbounds="locations", visible=False)
        fig.update_layout(width=1000, height=700, margin={"r":0, "t":80, "l":0, "b":0}, title_x=0.5)
        fig.show()

# --- 4. ATTACH THE OBSERVERS ---
# This tells the code to run update_map whenever a dropdown changes
level_dropdown.observe(update_map, names='value')
metric_dropdown.observe(update_map, names='value')

# --- 5. INITIAL DISPLAY ---
print("🎛️ ADHAAR PROJECT: INTERACTIVE RISK DASHBOARD")
display(widgets.HBox([level_dropdown, metric_dropdown]))
display(out)
update_map(None) # Call it once to show the first map

## Aadhar Enrolment Over Time

In [ ]:
df_master['date'] = pd.to_datetime(df_master['date'],dayfirst=True)
df_master['year_month'] = df_master['date'].dt.to_period('M')
enrol_time = df_master.groupby('year_month').size().reset_index(name='enrolment_count')
enrol_time['year_month'] = enrol_time['year_month'].astype(str)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --- 1. Data Preparation (Calculations) ---
# Enrolment by State
enrol_state = df_master.groupby("state").size().sort_values(ascending=False).head(10)

# Update Type Distribution
bio_total = df_master1["bio_age_5_17"].sum() + df_master1["bio_age_17_"].sum()
demo_total = df_master2["demo_age_5_17"].sum() + df_master2["demo_age_17_"].sum()

# Enrolment Age Groups
enrol_age_series = pd.Series({
    "0-5 Yrs": df_master["age_0_5"].sum(),
    "5-17 Yrs": df_master["age_5_17"].sum(),
    "18+ Yrs": df_master["age_18_greater"].sum()
})

# Update Age Groups
update_age_series = pd.Series({
    "5-17 (Bio)": df_master1["bio_age_5_17"].sum(),
    "17+ (Demo)": df_master2["demo_age_17_"].sum()
})

# --- 2. Dashboard Plotting ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12)) # Create a 2x2 grid
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# Subplot 1: Total Enrolment by State
enrol_state.plot(kind="bar", ax=axes[0, 0], color='teal', edgecolor='black')
axes[0, 0].set_title("Top 10 States by Enrolment", fontweight='bold')
axes[0, 0].set_ylabel("Enrolments")

# Subplot 2: Update Type Distribution
axes[0, 1].bar(["Demographic", "Biometric"], [demo_total, bio_total], color=['#66b3ff', '#ff9999'])
axes[0, 1].set_title("Update Type Distribution", fontweight='bold')
axes[0, 1].set_ylabel("Update Count")

# Subplot 3: Enrolments by Age Group
enrol_age_series.plot(kind="bar", ax=axes[1, 0], color=['#99ff99', '#66b3ff', '#ffcc99'], edgecolor='black')
axes[1, 0].set_title("Enrolments by Age Group", fontweight='bold')
axes[1, 0].set_ylabel("Total People")
axes[1, 0].tick_params(axis='x', rotation=0)

# Subplot 4: Updates by Age Group
update_age_series.plot(kind="bar", ax=axes[1, 1], color=['skyblue', 'steelblue'], edgecolor='black')
axes[1, 1].set_title("Updates by Age Category", fontweight='bold')
axes[1, 1].set_ylabel("Total Updates")
axes[1, 1].tick_params(axis='x', rotation=0)

# Final Cleanup
plt.suptitle("📊 Comprehensive Aadhaar Enrolment & Update Insights", fontsize=18, y=0.98, fontweight='bold')
plt.show()

## Update Rate
Update Rate is the ratio of Aadhaar updates to total enrolments in a given state and time period.

Lower values may indicate identity maintenance backlog or disengagement.

In [ ]:
enrol_state = df_master.groupby("state")[['age_0_5', 'age_5_17', 'age_18_greater']].sum().sum(axis=1).to_frame("total_enrolments")
upd_state = df_master2.groupby("state")[['demo_age_5_17', 'demo_age_17_']].sum().sum(axis=1).to_frame("updates")

engagement = enrol_state.join(upd_state, how="inner").fillna(0)

# Robust Update Rate calculation (Fix 2)
engagement["update_rate"] = (
    engagement["updates"] / engagement["total_enrolments"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

## Update Lag

In [ ]:

# 2. ANALYZE: Create the state_summary for Update Lag
state_summary = (
    engagement
    .groupby("state")
    .agg(
        total_enrolments=("total_enrolments", "sum"),
        avg_update_rate=("update_rate", "mean")
    )
    .reset_index()
)

# 3. FLAG: Apply the binary risk signal
state_summary["update_lag_flag"] = np.where(
    (state_summary["total_enrolments"] > state_summary["total_enrolments"].median()) &
    (state_summary["avg_update_rate"] < state_summary["avg_update_rate"].median()),
    "High Risk",
    "Normal"
)

# 4. VIEW: Show the High Risk states
print("States Flagged as High Risk (High Volume, Low Update Activity):")

print(state_summary[state_summary["update_lag_flag"] == "High Risk"])

In [ ]:
upd_state = df_master2.groupby("state")[["demo_age_5_17", "demo_age_17_"]].sum()

# 2. Sort by total updates so the heatmap looks organized (highest at top)
upd_state['total'] = upd_state.sum(axis=1)
upd_state = upd_state.sort_values("total", ascending=False).drop(columns='total')

# 3. Create the Heatmap
plt.figure(figsize=(10, 10))
sns.heatmap(
    upd_state.head(20), # Showing top 20 states for clarity
    cmap="Blues", 
    annot=True,          # Shows the exact numbers in the boxes
    fmt=",d",            # Adds commas to numbers (e.g., 1,250)
    linewidths=0.5
)

plt.title("Demographic Updates by State and Age Group", fontsize=15)
plt.xlabel("Age Group")
plt.ylabel("State")
plt.savefig('plots/demographic_updates_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_master['total_enrol_row'] = df_master[['age_0_5', 'age_5_17', 'age_18_greater']].sum(axis=1)

# 2. Now group by State and sum that new total column
enrol_state = (
    df_master.groupby("state")["total_enrol_row"]
    .sum()
    .to_frame("enrolments")
    .sort_values(by="enrolments", ascending=False)
)

# 3. Calculate the normalized value (0.0 to 1.0)
# This identifies which states are closest to the leader (likely UP or Maharashtra)
enrol_state["enrol_norm"] = (
    enrol_state["enrolments"] / enrol_state["enrolments"].max()
)

print(enrol_state.head(10))

In [ ]:
df_master2['update_count'] = df_master2[['demo_age_5_17', 'demo_age_17_']].sum(axis=1)

updates_state = (
    df_master2.groupby("state")["update_count"]
    .sum()
    .to_frame("total_updates")
)

# 2. Prepare Enrollment Data (enrol)
# Using the sum(axis=1) logic for 'age 0-5', '5-17', and '18+'
df_master['enrolment_count'] = df_master[['age_0_5', 'age_5_17', 'age_18_greater']].sum(axis=1)

enrol_state_summary = (
    df_master.groupby("state")["enrolment_count"]
    .sum()
    .to_frame("total_enrolments")
)

# 3. Join the datasets
# We join 'updates_state' into our 'enrol_state_summary'
engagement = enrol_state_summary.join(updates_state, how="left").fillna(0)

# 4. Calculate the Update Rate
# This tells you: "For every person enrolled in this state, how many updates occurred?"
engagement["update_rate"] = engagement["total_updates"] / engagement["total_enrolments"]

# Sort by the highest update rate to find the most active states
engagement = engagement.sort_values(by="update_rate", ascending=False)

print(engagement.head(10))

In [ ]:
demo_long = df_master2.melt(
    id_vars=['state'], 
    value_vars=['demo_age_5_17', 'demo_age_17_'], 
    var_name='age_group', 
    value_name='value_count' # Changed this name to avoid the conflict
)

# 2. Calculate Totals per State/Age Group
demo_rates = demo_long.groupby(["state", "age_group"])["value_count"].sum().reset_index()

# 3. Calculate Variance per State
variance = demo_rates.groupby("state")["value_count"].var().fillna(0)

# 4. Calculate the Balance Score
demo_balance = (1 / (1 + variance)).to_frame("balance_score")
balance = 1 / (1 + variance)

# 5. Calculate the Total Updates per State (The "Permanent" Column logic)
# Now we create the state-level totals to join with our balance score
state_totals = demo_rates.groupby("state")["value_count"].sum().to_frame("total_updates")

# 6. Final Merge
final_analysis = state_totals.join(demo_balance)

print(final_analysis.sort_values(by="balance_score", ascending=False).head(10))

In [ ]:
# Single-source-of-truth state-level engagement (replaces the three duplicate
# rebuilds previously in cells 32, 33, 34). All downstream cells consume `engagement`.
engagement = build_state_engagement(df_master, df_master2)

# State-level risk index (re-uses district-level helper for consistency)
state_risk = calculate_risk_index(engagement.reset_index())
state_risk = state_risk[state_risk["total_enrolments"] > CONFIG["thresholds"]["min_enrol_for_analysis"]].copy()
state_risk = state_risk.sort_values("identity_maintenance_risk", ascending=False)

print("Top 10 States by Identity Maintenance Risk:")
print(state_risk[["state", "total_enrolments", "p_failure", "impact",
                  "identity_maintenance_risk", "risk_level"]].head(10).to_string(index=False))

# Inclusion-Index leaderboard (positive-framing of the same components)
state_risk["inclusion_index"] = 1.0 - state_risk["identity_maintenance_risk"]
top15 = state_risk.nlargest(15, "inclusion_index")
plt.figure(figsize=(12, 8))
ax = sns.barplot(data=top15, x="inclusion_index", y="state", palette="viridis")
for p in ax.patches:
    ax.text(p.get_width() + 0.005, p.get_y() + p.get_height() / 2,
            f"{p.get_width():.3f}", va="center", fontsize=11, fontweight="bold")
plt.title("Aadhaar Inclusion Index: Top 15 States", fontsize=18, pad=20, fontweight="bold")
plt.xlabel("Inclusion Index (Weighted Score)"); plt.ylabel("State")
sns.despine(left=True, bottom=True); plt.tight_layout(); plt.show()


In [ ]:
# Tertile classification already lives in `state_risk['risk_level']`. Show it.
print("\nState-level Risk Tiers:")
print(state_risk.groupby("risk_level")["state"].apply(list))


In [ ]:
# Real sensitivity analysis: bootstrap weight perturbations.
state_sens = sensitivity_rank_stability(state_risk, n_perturbations=200, weight_jitter=0.15)
state_sens["state"] = state_risk["state"].values
print("\nState-level rank stability under ±15% weight jitter:")
print(state_sens.sort_values("rank_mean").head(10))


## Policy Implications
States classified as High Risk exhibit high Aadhaar enrolments but relatively low update activity, indicating potential identity maintenance backlog. UIDAI can prioritize update outreach programs, improve accessibility of update centers, and conduct targeted awareness campaigns in these regions to mitigate future authentication failures and service exclusion.